<a href="https://colab.research.google.com/github/Rohil121/bharat-portfolio-lab/blob/v0.4-forecasting-risk-models/notebooks/04_forecasting_risk_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bharat Portfolio Lab v0.4 — Forecasting and Risk Models

## Objective

This notebook develops forecasting and volatility-risk models for Indian
listed equities.

The v0.4 framework will include:

- ARIMA return forecasting
- naive benchmark forecasts
- forecast-confidence intervals
- stationarity and residual diagnostics
- ARCH/GARCH volatility forecasting
- forecast evaluation using a chronological test period
- dynamic volatility forecasts for portfolio allocation

## Research principle

Forecasting models will be evaluated against simple benchmarks rather than
assumed to be useful.

The objective is to quantify uncertainty and forecast risk, not to claim
that tomorrow's stock price can be predicted accurately.

## Dynamic-portfolio requirement

All forecasting functions must accept arbitrary NSE/BSE ticker inputs so
that they can later support user-selected portfolios in the final
application.

In [1]:
%pip install -q yfinance statsmodels arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 13.0 MB/s eta 0:00:00


In [2]:
# ---------------------------------------------------------
# v0.4 environment setup
# ---------------------------------------------------------

from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

from arch import arch_model
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)

from statsmodels.graphics.tsaplots import (
    plot_acf,
    plot_pacf,
)

from statsmodels.stats.diagnostic import (
    acorr_ljungbox,
    het_arch,
)

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")

TRADING_DAYS = 252
RISK_FREE_RATE = 0.065

DEFAULT_BENCHMARK_TICKER = "^NSEI"
DEFAULT_BENCHMARK_NAME = "Nifty 50"

DEFAULT_FORECAST_TICKERS = [
    "HDFCBANK.NS",
    "TCS.NS",
    "HINDUNILVR.NS",
    "SUNPHARMA.NS",
    "POWERGRID.NS",
    "BHARTIARTL.NS",
    "LT.NS",
    "M&M.NS",
    "BEL.NS",
    "TRENT.NS",
]

FORECAST_DATA_START = "2016-01-01"
FORECAST_TEST_START = pd.Timestamp("2024-01-01")

print("v0.4 forecasting environment prepared.")
print("Default equities:", len(DEFAULT_FORECAST_TICKERS))
print("Forecast test period begins:", FORECAST_TEST_START.date())

v0.4 forecasting environment prepared.
Default equities: 10
Forecast test period begins: 2024-01-01


## 1. Market Data and Chronological Split

Daily adjusted closing prices are downloaded for the selected Indian
equities and benchmark.

Forecasting models will be trained using observations before
1 January 2024 and evaluated only on observations from 1 January 2024
onward.

Daily log returns are used for return forecasting because stock-price
levels are generally non-stationary.

The data function accepts arbitrary ticker lists so it can later support
user-selected portfolios in the Streamlit application.

In [3]:
# ---------------------------------------------------------
# Reusable Indian market-data loader
# ---------------------------------------------------------

def download_adjusted_close_prices(
    tickers,
    benchmark_ticker="^NSEI",
    start_date="2016-01-01",
    end_date=None,
):
    """
    Download adjusted closing prices for an arbitrary list of
    NSE/BSE securities and an Indian market benchmark.
    """

    if not tickers:
        raise ValueError(
            "At least one equity ticker must be supplied."
        )

    cleaned_tickers = [
        str(ticker).strip().upper()
        for ticker in tickers
        if str(ticker).strip()
    ]

    all_tickers = list(
        dict.fromkeys(
            cleaned_tickers
            + [benchmark_ticker]
        )
    )

    if end_date is None:
        end_date = (
            pd.Timestamp.today().normalize()
            + pd.Timedelta(days=1)
        )

    raw_data = yf.download(
        tickers=all_tickers,
        start=str(pd.Timestamp(start_date).date()),
        end=str(pd.Timestamp(end_date).date()),
        auto_adjust=True,
        progress=False,
        threads=True,
    )

    if raw_data.empty:
        raise ValueError(
            "No market data was returned."
        )

    # Handle Yahoo Finance's possible column structures
    if isinstance(raw_data.columns, pd.MultiIndex):

        if "Close" in raw_data.columns.get_level_values(0):

            close_prices = (
                raw_data["Close"]
                .copy()
            )

        elif "Close" in raw_data.columns.get_level_values(1):

            close_prices = (
                raw_data
                .xs(
                    "Close",
                    axis=1,
                    level=1,
                )
                .copy()
            )

        else:
            raise ValueError(
                "Closing-price data was not returned."
            )

    else:

        if "Close" not in raw_data.columns:
            raise ValueError(
                "Closing-price data was not returned."
            )

        close_prices = raw_data[["Close"]].copy()

        if len(all_tickers) == 1:
            close_prices.columns = all_tickers

    close_prices = (
        close_prices
        .reindex(columns=all_tickers)
        .sort_index()
        .dropna(how="all")
    )

    missing_tickers = [
        ticker
        for ticker in all_tickers
        if (
            ticker not in close_prices.columns
            or close_prices[ticker].dropna().empty
        )
    ]

    if missing_tickers:
        raise ValueError(
            "No usable price history was found for: "
            + ", ".join(missing_tickers)
        )

    return close_prices

In [4]:
# ---------------------------------------------------------
# Download forecasting dataset
# ---------------------------------------------------------

forecast_price_data = download_adjusted_close_prices(
    tickers=DEFAULT_FORECAST_TICKERS,
    benchmark_ticker=DEFAULT_BENCHMARK_TICKER,
    start_date=FORECAST_DATA_START,
)

forecast_stock_prices = (
    forecast_price_data[
        DEFAULT_FORECAST_TICKERS
    ]
    .copy()
)

forecast_benchmark_prices = (
    forecast_price_data[
        DEFAULT_BENCHMARK_TICKER
    ]
    .rename(DEFAULT_BENCHMARK_NAME)
)

print("Forecasting market data downloaded.")
print("-" * 60)
print(
    "Available period:",
    forecast_price_data.index.min().date(),
    "to",
    forecast_price_data.index.max().date(),
)
print(
    "Securities:",
    len(DEFAULT_FORECAST_TICKERS),
)
print(
    "Benchmark:",
    DEFAULT_BENCHMARK_NAME,
)

Forecasting market data downloaded.
------------------------------------------------------------
Available period: 2016-01-01 to 2026-07-30
Securities: 10
Benchmark: Nifty 50


In [5]:
# ---------------------------------------------------------
# Calculate log returns and chronological split
# ---------------------------------------------------------

forecast_log_returns = (
    np.log(
        forecast_price_data
        / forecast_price_data.shift(1)
    )
)

development_log_returns = (
    forecast_log_returns.loc[
        forecast_log_returns.index
        < FORECAST_TEST_START
    ]
)

test_log_returns = (
    forecast_log_returns.loc[
        forecast_log_returns.index
        >= FORECAST_TEST_START
    ]
)

assert not development_log_returns.empty
assert not test_log_returns.empty
assert (
    development_log_returns.index.max()
    < FORECAST_TEST_START
)
assert (
    test_log_returns.index.min()
    >= FORECAST_TEST_START
)

print("Chronological forecasting split completed.")
print("-" * 60)
print(
    "Development period:",
    development_log_returns.index.min().date(),
    "to",
    development_log_returns.index.max().date(),
)
print(
    "Test period:",
    test_log_returns.index.min().date(),
    "to",
    test_log_returns.index.max().date(),
)

Chronological forecasting split completed.
------------------------------------------------------------
Development period: 2016-01-01 to 2023-12-29
Test period: 2024-01-01 to 2026-07-30


In [6]:
# ---------------------------------------------------------
# Data-quality summary by security
# ---------------------------------------------------------

data_quality_records = []

for ticker in forecast_price_data.columns:

    ticker_prices = (
        forecast_price_data[ticker]
        .dropna()
    )

    ticker_development_returns = (
        development_log_returns[ticker]
        .dropna()
    )

    ticker_test_returns = (
        test_log_returns[ticker]
        .dropna()
    )

    data_quality_records.append(
        {
            "Ticker": ticker,
            "First Price Date":
                ticker_prices.index.min().date(),

            "Latest Price Date":
                ticker_prices.index.max().date(),

            "Price Observations":
                len(ticker_prices),

            "Development Returns":
                len(ticker_development_returns),

            "Test Returns":
                len(ticker_test_returns),

            "Missing Price Values":
                forecast_price_data[ticker]
                .isna()
                .sum(),
        }
    )


forecast_data_quality = (
    pd.DataFrame(data_quality_records)
    .set_index("Ticker")
)

display(forecast_data_quality)

assert (
    forecast_data_quality[
        "Development Returns"
    ] >= 500
).all()

assert (
    forecast_data_quality[
        "Test Returns"
    ] >= 100
).all()

,First Price Date,Latest Price Date,Price Observations,Development Returns,Test Returns,Missing Price Values
Ticker,,,,,,
HDFCBANK.NS,2016-01-01,2026-07-30,2615,1974,640,0
TCS.NS,2016-01-01,2026-07-30,2615,1974,640,0
HINDUNILVR.NS,2016-01-01,2026-07-30,2615,1974,640,0
SUNPHARMA.NS,2016-01-01,2026-07-30,2615,1974,640,0
POWERGRID.NS,2016-01-01,2026-07-30,2615,1974,640,0
BHARTIARTL.NS,2016-01-01,2026-07-30,2615,1974,640,0
LT.NS,2016-01-01,2026-07-30,2615,1974,640,0
M&M.NS,2016-01-01,2026-07-30,2615,1974,640,0
BEL.NS,2016-01-01,2026-07-30,2615,1974,640,0


## 2. Return-Series Stationarity Diagnostics

ARIMA models require the modelled time series to be reasonably
stationary.

A stationary series has statistical properties such as its mean and
variance that remain broadly stable through time.

Daily stock-price levels are usually non-stationary. Daily log returns
are generally more suitable for ARIMA modelling.

The Augmented Dickey–Fuller test is applied only to development-period
returns.

### Hypotheses

- **Null hypothesis:** the series contains a unit root and is non-stationary
- **Alternative hypothesis:** the series is stationary

A p-value below 5% leads to rejection of the null hypothesis.

In [7]:
# ---------------------------------------------------------
# Reusable Augmented Dickey–Fuller diagnostic
# ---------------------------------------------------------

def run_adf_stationarity_test(
    time_series,
    significance_level=0.05,
):
    """
    Run the Augmented Dickey–Fuller stationarity test
    on one return series.
    """

    clean_series = (
        pd.Series(time_series)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
    )

    if len(clean_series) < 100:
        raise ValueError(
            "At least 100 observations are required "
            "for the stationarity test."
        )

    adf_result = adfuller(
        clean_series,
        autolag="AIC",
    )

    adf_statistic = adf_result[0]
    p_value = adf_result[1]
    selected_lags = adf_result[2]
    observations = adf_result[3]
    critical_values = adf_result[4]

    is_stationary = (
        p_value < significance_level
    )

    return {
        "ADF Statistic": adf_statistic,
        "P-Value": p_value,
        "Selected Lags": selected_lags,
        "Observations": observations,
        "1% Critical Value": critical_values["1%"],
        "5% Critical Value": critical_values["5%"],
        "10% Critical Value": critical_values["10%"],
        "Stationary at 5%": is_stationary,
    }

In [8]:
# ---------------------------------------------------------
# Development-period stationarity testing
# ---------------------------------------------------------

stationarity_records = []

for ticker in development_log_returns.columns:

    ticker_result = run_adf_stationarity_test(
        development_log_returns[ticker]
    )

    stationarity_records.append(
        {
            "Ticker": ticker,
            **ticker_result,
        }
    )

stationarity_results = (
    pd.DataFrame(stationarity_records)
    .set_index("Ticker")
)

print("Development-Period ADF Stationarity Results")
display(
    stationarity_results.style.format(
        {
            "ADF Statistic": "{:.4f}",
            "P-Value": "{:.6f}",
            "1% Critical Value": "{:.4f}",
            "5% Critical Value": "{:.4f}",
            "10% Critical Value": "{:.4f}",
        }
    )
)

Development-Period ADF Stationarity Results


,ADF Statistic,P-Value,Selected Lags,Observations,1% Critical Value,5% Critical Value,10% Critical Value,Stationary at 5%
Ticker,,,,,,,,
HDFCBANK.NS,-9.9991,0.000000,21,1952,-3.4337,-2.8630,-2.5676,True
TCS.NS,-45.1341,0.000000,0,1973,-3.4337,-2.8630,-2.5676,True
HINDUNILVR.NS,-15.9853,0.000000,6,1967,-3.4337,-2.8630,-2.5676,True
SUNPHARMA.NS,-44.8964,0.000000,0,1973,-3.4337,-2.8630,-2.5676,True
POWERGRID.NS,-19.3835,0.000000,6,1967,-3.4337,-2.8630,-2.5676,True
BHARTIARTL.NS,-33.2717,0.000000,1,1972,-3.4337,-2.8630,-2.5676,True
LT.NS,-14.2411,0.000000,8,1965,-3.4337,-2.8630,-2.5676,True
M&M.NS,-17.1088,0.000000,5,1968,-3.4337,-2.8630,-2.5676,True
BEL.NS,-16.3297,0.000000,6,1967,-3.4337,-2.8630,-2.5676,True


In [9]:
stationary_series_count = int(
    stationarity_results[
        "Stationary at 5%"
    ].sum()
)

total_series_count = len(
    stationarity_results
)

stationarity_summary = pd.DataFrame(
    {
        "Result": [
            total_series_count,
            stationary_series_count,
            total_series_count
            - stationary_series_count,
            stationary_series_count
            / total_series_count,
        ],
    },
    index=[
        "Series Tested",
        "Stationary at 5%",
        "Non-Stationary at 5%",
        "Stationary Percentage",
    ],
)

formatted_stationarity_summary = (
    stationarity_summary
    .copy()
    .astype(object)
)

formatted_stationarity_summary.loc[
    "Stationary Percentage",
    "Result",
] = (
    f"{stationary_series_count / total_series_count:.2%}"
)

print("Stationarity Summary")
display(formatted_stationarity_summary)

Stationarity Summary


,Result
Series Tested,11.0
Stationary at 5%,11.0
Non-Stationary at 5%,0.0
Stationary Percentage,100.00%


## 3. ARIMA Order Selection

Because all development-period log-return series passed the stationarity
test, the integration order is fixed at zero.

Candidate ARIMA(p, 0, q) models are compared using only development-period
data.

The preferred order is selected using the Bayesian Information Criterion
(BIC). BIC rewards goodness of fit but penalises unnecessary model
complexity more strongly than AIC.

No observations from the out-of-sample test period are used during model
selection.

In [10]:
# ---------------------------------------------------------
# Reusable ARIMA order-selection function
# ---------------------------------------------------------

def select_arima_order(
    time_series,
    p_values=range(0, 4),
    q_values=range(0, 4),
    selection_criterion="BIC",
):
    """
    Compare ARIMA(p, 0, q) models using development data only.

    Returns:
        selected_order
        ranked_results
        fitted_models
    """

    clean_series = (
        pd.Series(time_series)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
    )

    if len(clean_series) < 500:
        raise ValueError(
            "At least 500 observations are required "
            "for ARIMA order selection."
        )

    # Percentage scaling improves numerical optimisation.
    scaled_series = clean_series * 100

    selection_records = []
    fitted_models = {}

    for p_value in p_values:

        for q_value in q_values:

            model_order = (
                int(p_value),
                0,
                int(q_value),
            )

            try:

                with warnings.catch_warnings():

                    warnings.simplefilter(
                        "ignore"
                    )

                    fitted_model = ARIMA(
                        scaled_series,
                        order=model_order,
                        trend="c",
                        enforce_stationarity=False,
                        enforce_invertibility=False,
                    ).fit()

                converged = bool(
                    fitted_model.mle_retvals.get(
                        "converged",
                        True,
                    )
                )

                selection_records.append(
                    {
                        "Order": model_order,
                        "p": p_value,
                        "d": 0,
                        "q": q_value,
                        "AIC": fitted_model.aic,
                        "BIC": fitted_model.bic,
                        "HQIC": fitted_model.hqic,
                        "Log Likelihood": fitted_model.llf,
                        "Parameters": len(
                            fitted_model.params
                        ),
                        "Converged": converged,
                    }
                )

                fitted_models[
                    model_order
                ] = fitted_model

            except Exception as error:

                selection_records.append(
                    {
                        "Order": model_order,
                        "p": p_value,
                        "d": 0,
                        "q": q_value,
                        "AIC": np.nan,
                        "BIC": np.nan,
                        "HQIC": np.nan,
                        "Log Likelihood": np.nan,
                        "Parameters": np.nan,
                        "Converged": False,
                        "Error": str(error),
                    }
                )

    order_results = pd.DataFrame(
        selection_records
    )

    criterion_column = (
        selection_criterion
        .strip()
        .upper()
    )

    allowed_criteria = {
        "AIC",
        "BIC",
        "HQIC",
    }

    if criterion_column not in allowed_criteria:
        raise ValueError(
            "Selection criterion must be AIC, BIC or HQIC."
        )

    valid_results = (
        order_results.loc[
            order_results["Converged"]
            & order_results[
                criterion_column
            ].notna()
        ]
        .sort_values(
            criterion_column,
            ascending=True,
        )
        .reset_index(drop=True)
    )

    if valid_results.empty:
        raise RuntimeError(
            "No ARIMA candidate converged successfully."
        )

    selected_order = tuple(
        valid_results.loc[
            0,
            ["p", "d", "q"],
        ].astype(int)
    )

    return {
        "Selected Order": selected_order,
        "Ranked Results": valid_results,
        "All Results": order_results,
        "Fitted Models": fitted_models,
        "Selection Criterion": criterion_column,
    }

In [11]:
# ---------------------------------------------------------
# Nifty 50 development-period ARIMA selection
# ---------------------------------------------------------

benchmark_arima_selection = select_arima_order(
    time_series=development_log_returns[
        DEFAULT_BENCHMARK_TICKER
    ],
    p_values=range(0, 4),
    q_values=range(0, 4),
    selection_criterion="BIC",
)

benchmark_arima_order = (
    benchmark_arima_selection[
        "Selected Order"
    ]
)

benchmark_arima_ranking = (
    benchmark_arima_selection[
        "Ranked Results"
    ]
)

print("Nifty 50 ARIMA Order Selection")
print("-" * 60)
print(
    "Selected order:",
    benchmark_arima_order,
)
print(
    "Selection criterion:",
    benchmark_arima_selection[
        "Selection Criterion"
    ],
)
print(
    "Successful candidate models:",
    len(benchmark_arima_ranking),
)

display(
    benchmark_arima_ranking[
        [
            "Order",
            "AIC",
            "BIC",
            "HQIC",
            "Log Likelihood",
            "Parameters",
        ]
    ]
    .head(10)
    .style.format(
        {
            "AIC": "{:.2f}",
            "BIC": "{:.2f}",
            "HQIC": "{:.2f}",
            "Log Likelihood": "{:.2f}",
            "Parameters": "{:.0f}",
        }
    )
)

Nifty 50 ARIMA Order Selection
------------------------------------------------------------
Selected order: (2, 0, 3)
Selection criterion: BIC
Successful candidate models: 15


,Order,AIC,BIC,HQIC,Log Likelihood,Parameters
0,"(2, 0, 3)",5799.69,5838.75,5814.05,-2892.84,7
1,"(3, 0, 3)",5802.79,5847.43,5819.20,-2893.40,8
2,"(0, 0, 0)",5856.33,5867.49,5860.43,-2926.16,2
3,"(0, 0, 1)",5854.66,5871.40,5860.81,-2924.33,3
4,"(0, 0, 2)",5849.35,5871.67,5857.55,-2920.67,4
5,"(1, 0, 3)",5839.20,5872.68,5851.51,-2913.60,6
6,"(1, 0, 0)",5856.98,5873.73,5863.14,-2925.49,3
7,"(3, 0, 1)",5840.92,5874.41,5853.23,-2914.46,6
8,"(0, 0, 3)",5848.83,5876.73,5859.09,-2919.42,5
9,"(1, 0, 1)",5855.89,5878.21,5864.09,-2923.94,4
